# 04 — Backtest Results

Run the mean-reversion strategy on ERCOT-PJM spread with walk-forward validation.
Walk-forward prevents overfitting by training on a rolling window and testing out-of-sample.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data.fetcher import ISODataFetcher
from src.analysis.spreads import SpreadAnalyzer
from src.strategy.mean_reversion import MeanReversionStrategy
from src.strategy.backtest import BacktestEngine
from src.strategy.optimize import StrategyOptimizer

In [ ]:
# Load data
fetcher = ISODataFetcher(config_path='../config/settings.yaml')
analyzer = SpreadAnalyzer()

end_date = pd.Timestamp.now().strftime('%Y-%m-%d')
start_date = (pd.Timestamp.now() - pd.Timedelta(days=365)).strftime('%Y-%m-%d')

ercot = fetcher.fetch('ERCOT', start_date, end_date)
pjm = fetcher.fetch('PJM', start_date, end_date)
spread_df = analyzer.compute_spread(ercot, pjm)
spreads = spread_df['spread']

In [ ]:
# Run mean-reversion strategy
strategy = MeanReversionStrategy(lookback=20, entry_z=1.5, exit_z=0.3, stop_loss_z=3.0)
signals = strategy.generate_signals(spreads)

# Backtest
engine = BacktestEngine()
result = engine.run(signals, initial_capital=100_000, position_size_mw=50)

print("Backtest Results:")
for k, v in result['metrics'].items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# Equity curve
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['Equity Curve', 'Position'])

fig.add_trace(go.Scatter(y=result['equity_curve'], name='Equity',
                         line=dict(color='green')), row=1, col=1)
fig.add_trace(go.Scatter(y=signals['position'], name='Position',
                         line=dict(color='blue')), row=2, col=1)

fig.update_layout(height=600, title_text='Mean Reversion Strategy — ERCOT/PJM')
fig.show()

In [ ]:
# Walk-forward validation
wf_result = engine.walk_forward(
    strategy_class=MeanReversionStrategy,
    strategy_params={'lookback': 20, 'entry_z': 1.5, 'exit_z': 0.3, 'stop_loss_z': 3.0},
    spreads=spreads,
    train_window=60,
    test_window=30,
)

print("Walk-Forward Results:")
print(f"  Number of folds: {wf_result['n_folds']}")
for k, v in wf_result['overall_metrics'].items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

print("\nFold-by-fold:")
wf_result['fold_results']

In [ ]:
# Parameter sensitivity — how does Sharpe vary with entry_z?
optimizer = StrategyOptimizer()
sensitivity = optimizer.sensitivity_analysis(
    strategy_class=MeanReversionStrategy,
    spreads=spreads,
    base_params={'lookback': 20, 'exit_z': 0.3, 'stop_loss_z': 3.0},
    param_name='entry_z',
    param_range=[0.5, 1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0],
)

fig = px.line(sensitivity, x='entry_z', y='sharpe',
              title='Parameter Sensitivity: Entry Z-Score vs Sharpe Ratio',
              markers=True)
fig.show()

## Limitations

1. **Synthetic data**: Results based on simulated prices — real market dynamics may differ
2. **Transaction costs**: Modeled as flat \$0.05/MWh; real costs include bid-ask spread, slippage
3. **No capacity constraints**: Assumes unlimited position sizes
4. **Single pair**: Production would trade a portfolio of spread pairs
5. **No regime awareness**: Strategy doesn't adjust parameters during volatile regimes